In [13]:
from tcc import import_macro_series, import_bitcoin_only, get_bitcoin_date_range
import warnings
warnings.filterwarnings('ignore')

# Ver período disponível
start, end = get_bitcoin_date_range()
print(f"Bitcoin disponível de {start} a {end}")

# Carregar dados para análise
data = import_macro_series(["vix", "creditSpreads", "fedFundsRate", 
                            'realGDP', 'treasury10Y2YSpread'])
print(data.head())

Bitcoin disponível de 2013-04-26 00:00:00 a 2025-06-15 00:00:00
Ajustando dados para período do Bitcoin: 2013-04-26 a 2025-06-15
Bitcoin carregado: 4432 observações
Série 'vix' carregada: 4431 observações
Série 'creditSpreads' carregada: 4431 observações
Série 'fedFundsRate' carregada: 4430 observações
Série 'realGDP' carregada: 4269 observações
Série 'treasury10Y2YSpread' carregada: 4305 observações
            bitcoin    vix  creditSpreads  fedFundsRate    realGDP  \
2013-04-26      NaN  13.61           4.60          0.15  17709.671   
2013-04-27      NaN  13.61           4.60          0.15  17709.671   
2013-04-28   135.30  13.61           4.60          0.15  17709.671   
2013-04-29   141.96  13.71           4.53          0.15  17709.671   
2013-04-30   135.30  13.52           4.55          0.15  17709.671   

            treasury10Y2YSpread  
2013-04-26                 1.48  
2013-04-27                 1.48  
2013-04-28                 1.48  
2013-04-29                 1.50  
2013-

In [14]:
# Filtrar dados para período entre 2017 e 2023
data_filtered = data[(data.index >= '2017-01-01') & (data.index <= '2023-12-31')]

# Criar coluna de log retornos do bitcoin
import numpy as np
data_filtered = data_filtered.copy()
data_filtered['bitcoin_log_returns'] = np.log(data_filtered['bitcoin'] / data_filtered['bitcoin'].shift(1))

# Remover primeira linha que terá NaN devido ao shift
data_filtered = data_filtered.dropna()

data_filtered



,bitcoin,vix,creditSpreads,fedFundsRate,realGDP,treasury10Y2YSpread,bitcoin_log_returns
2017-01-02,1019.20,14.04,4.22,0.66,19398.343,1.25,0.020970
2017-01-03,1035.53,12.85,4.13,0.66,19398.343,1.23,0.015895
2017-01-04,1130.85,11.85,4.02,0.60,19398.343,1.22,0.088056
2017-01-05,990.67,11.67,4.04,0.60,19398.343,1.20,-0.132343
2017-01-06,894.03,11.32,3.98,0.60,19398.343,1.20,-0.102642
...,...,...,...,...,...,...,...
2023-12-27,43418.00,12.43,3.34,5.33,22960.600,-0.41,0.020994
2023-12-28,42601.00,12.47,3.32,5.33,22960.600,-0.42,-0.018996
2023-12-29,42075.00,12.45,3.34,5.33,22960.600,-0.35,-0.012424
2023-12-30,42221.00,12.45,3.34,5.33,22960.600,-0.35,0.003464


In [15]:
# Normalização dos dados usando RobustScaler

from sklearn.preprocessing import RobustScaler
import pandas as pd
import joblib

# Criar uma cópia dos dados para normalização
data_normalized = data_filtered.copy()

# Inicializar o RobustScaler
scaler = RobustScaler()

# Remover NaN antes da normalização
data_clean = data_normalized.dropna()

# Aplicar normalização apenas nas colunas numéricas (excluindo NaN)
# O RobustScaler é robusto a outliers e usa mediana e IQR
data_normalized_values = scaler.fit_transform(data_clean)

# Criar DataFrame normalizado mantendo índices e colunas corretos
data_normalized = pd.DataFrame(
    data_normalized_values, 
    index=data_clean.index, 
    columns=data_clean.columns
)

# Salvar os parâmetros do scaler para uso posterior em dados de teste
joblib.dump(scaler, 'tcc_scaler_parameters.pkl')
print("Parâmetros do scaler salvos em 'scaler_parameters.pkl'")

# Também salvar informações sobre as colunas usadas no treinamento
scaler_info = {
    'columns': list(data_clean.columns),
    'feature_names': list(data_clean.columns),
    'n_features': len(data_clean.columns)
}
joblib.dump(scaler_info, 'tcc_scaler_info.pkl')

print("Dados originais (últimas 5 linhas):")
print(data_filtered.tail())
print("\nDados normalizados (últimas 5 linhas):")
print(data_normalized.tail())
print(f"\nShape dos dados normalizados: {data_normalized.shape}")
print(f"Colunas utilizadas no scaler: {scaler_info['columns']}")


Parâmetros do scaler salvos em 'scaler_parameters.pkl'
Dados originais (últimas 5 linhas):
            bitcoin    vix  creditSpreads  fedFundsRate  realGDP  \
2023-12-27  43418.0  12.43           3.34          5.33  22960.6   
2023-12-28  42601.0  12.47           3.32          5.33  22960.6   
2023-12-29  42075.0  12.45           3.34          5.33  22960.6   
2023-12-30  42221.0  12.45           3.34          5.33  22960.6   
2023-12-31  42208.0  12.45           3.39          5.33  22960.6   

            treasury10Y2YSpread  bitcoin_log_returns  
2023-12-27                -0.41             0.020994  
2023-12-28                -0.42            -0.018996  
2023-12-29                -0.35            -0.012424  
2023-12-30                -0.35             0.003464  
2023-12-31                -0.35            -0.000308  

Dados normalizados (últimas 5 linhas):
             bitcoin       vix  creditSpreads  fedFundsRate   realGDP  \
2023-12-27  1.418237 -0.531767      -0.651163      1.6956

In [17]:
data_normalized.drop(columns=['bitcoin'], inplace=True)
data_normalized

In [21]:
array = data_normalized.to_numpy()
array

array([[-0.35985051,  0.37209302, -0.33478261, -0.85794432,  1.33823529,
         0.60706511],
       [-0.48691938,  0.26744186, -0.33478261, -0.85794432,  1.30882353,
         0.44902747],
       [-0.59369995,  0.13953488, -0.36086957, -0.85794432,  1.29411765,
         2.69634855],
       ...,
       [-0.52963161, -0.65116279,  1.69565217,  1.25711629, -1.01470588,
        -0.43292869],
       [-0.52963161, -0.65116279,  1.69565217,  1.25711629, -1.01470588,
         0.06187416],
       [-0.52963161, -0.59302326,  1.69565217,  1.25711629, -1.01470588,
        -0.05559619]])

In [22]:
from ripser import Rips
import persim
import matplotlib.pyplot as plt

rips = Rips(maxdim=2)

w = 50
n = len(data_normalized)-(2*w)+1

wasserstein_distances = np.zeros((n,1))

for i in range(n):
    
    dgm1 = rips.fit_transform(array[i:i+w])
    dgm2 = rips.fit_transform(array[i+w+1:i+(2*w)+1])

    wasserstein_distances[i] = persim.wasserstein(dgm1[0], dgm2[0], matching=False)









Rips(maxdim=2, thresh=inf, coeff=2, do_cocycles=False, n_perm = None, verbose=True)


In [37]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Criar subplots com plotly
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Preço do Bitcoin ao longo do tempo', 'Distância de Wasserstein ao longo do tempo'),
    vertical_spacing=0.1,
    shared_xaxes=True
)

# Subplot superior - Preços do Bitcoin
fig.add_trace(
    go.Scatter(
        x=data_filtered.index,
        y=data_filtered['bitcoin'],
        mode='lines',
        name='Bitcoin',
        line=dict(color='black', width=2.5)
    ),
    row=1, col=1
)

# Subplot inferior - Distâncias de Wasserstein
# Ajustar o índice para corresponder ao período das janelas deslizantes
start_idx = w
end_idx = start_idx + len(wasserstein_distances)
time_index = data_filtered.index[start_idx:end_idx]

fig.add_trace(
    go.Scatter(
        x=time_index,
        y=wasserstein_distances.flatten(),
        mode='lines',
        name='Distância de Wasserstein',
        line=dict(color='black', width=2.5, dash='dot')
    ),
    row=2, col=1
)

# Configurar layout
fig.update_layout(
    height=800,
    showlegend=False,
    title_text="",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black')
)

# Configurar eixos
fig.update_xaxes(title_text="", row=2, col=1, gridcolor='lightgray', linecolor='black')
fig.update_yaxes(title_text="Preço do Bitcoin", row=1, col=1, gridcolor='lightgray', linecolor='black')
fig.update_yaxes(title_text="Distância de Wasserstein", row=2, col=1, gridcolor='lightgray', linecolor='black')

fig.show()